# Phase 2 — Claim Extraction, Verdict, /verify, and Orchestration

Phase 2: claim extraction, verdict logic against the Phase 1 knowledge base,
`POST /verify`, and the Airflow DAG keeping the knowledge base fresh. Done —
see [docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md).

```mermaid
flowchart TB
    subgraph Verify["Verify pipeline — POST /verify"]
        direction TB
        TEXT[Report text] --> LLM[Claim Extractor LLM]
        LLM --> CLAIMS["Claims: entity, metric, value, date"]
        CLAIMS --> RAG[hybrid_search retrieval]
        RAG --> JUDGE[LLM-as-judge]
        JUDGE --> VERDICT["Verdict: VERIFIED / REFUTED / INSUFFICIENT + quote"]
        VERDICT --> API[POST /verify]
    end

    subgraph Orchestration["Airflow DAG — @daily"]
        direction LR
        FETCH["4x fetch_* tasks\n(wikipedia/worldbank/fred/secedgar)"] --> REBUILD[rebuild_vector_store]
    end

    REBUILD -.->|keeps fresh| RAG
```

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

## Claim extractor — structured output, not free text

`src/claim_extractor.py` asks the LLM for a typed `ClaimList` (`text`,
`entity`, `metric`, `value`, `date`), not free text to regex-parse. The
prompt drops opinions and forward-looking statements — only claims with an
entity and a number survive.

In [2]:
from src.claim_extractor import extract_claims

### Example

Three numeric claims (Apple revenue, Apple net income, Walmart revenue) mixed
with two opinion sentences that should be dropped.

In [3]:
report_text = """Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing."""

print(report_text)

Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing.


In [4]:
claims = extract_claims(report_text)
for c in claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000' entity='Apple Inc.' metric='Revenue' value=416161000000.0 date='2025-09-27'
text='net income came in at $112,010,000,000' entity='Apple Inc.' metric='Net income' value=112010000000.0 date='2025-09-27'
text='Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31' entity='Walmart' metric='Revenue' value=713163000000.0 date='2026-01-31'


Both opinion sentences were dropped correctly; all three claims have clean
`entity`/`metric`/`value`/`date` fields.

**Rough edge:** the first two claims share the same `text` — the model
reused the full sentence for both instead of the specific clause. Fields
are still correct.

## Model choice: OpenRouter default, Ollama and Groq as alternatives

Default: OpenRouter's free tier (`poolside/laguna-xs-2.1:free`, $0 cost).
Earlier defaults were dropped for provider-side issues — check
`LLM_PROVIDERS` in `src/config.py` for what's live now.

`LLM_PROVIDER=ollama` runs a local model (`ornith:latest`) — no API key or
limits, but CPU-only here. `LLM_PROVIDER=groq` (`qwen/qwen3.6-27b`) is a
fast third option, compared below.

In [5]:
from langchain_openai import ChatOpenAI
from src.claim_extractor import ClaimList, SYSTEM_PROMPT

ollama_llm = ChatOpenAI(
    model="ornith:latest",
    api_key="ollama",
    base_url="http://localhost:11434/v1",
    temperature=0,
).with_structured_output(ClaimList)

result = ollama_llm.invoke([("system", SYSTEM_PROMPT), ("user", report_text)])
for c in result.claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000' entity='Apple Inc.' metric='revenue' value=416161000000.0 date='fiscal year ending 2025-09-27'
text='net income came in at $112,010,000,000' entity='Apple Inc.' metric='net income' value=112010000000.0 date='fiscal year ending 2025-09-27'
text='Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31' entity='Walmart' metric='revenue' value=713163000000.0 date='fiscal year ending 2026-01-31'


Same three claims, correctly split — each `text` is the exact clause this
time, unlike the shared full sentence from OpenRouter's default above. Worth
A/B-testing models before locking in a default.

## Verdict — claim + retrieved evidence → VERIFIED / REFUTED / INSUFFICIENT

`verify_claim(claim, prompt=...)` in `src/verifier.py` embeds the claim,
runs `hybrid_search` (RRF over pgvector + full-text) against the knowledge
base, then has an LLM judge compare the claim to the top-5 chunks and
return a structured `Verdict`. `prompt` is a plain argument, so Phase 3 can
compare a second prompt on the same claims.

In [6]:
from src.claim_extractor import Claim
from src.verifier import verify_claim

### Three cases, one from each `data/eval_claims.csv` bucket

- `VERIFIED` — Apple revenue, correct value
- `REFUTED` — Apple revenue, wrong value (same source chunk as above)
- `INSUFFICIENT` — Netflix, not in the knowledge base at all

In [7]:
cases = [
    Claim(text="Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000.",
          entity="Apple", metric="Revenue", value=416161000000.0, date="2025-09-27"),
    Claim(text="Apple's revenue for fiscal year ending 2025-09-27 was $250,000,000,000.",
          entity="Apple", metric="Revenue", value=250000000000.0, date="2025-09-27"),
    Claim(text="Netflix's revenue for fiscal year 2025 was $39 billion.",
          entity="Netflix", metric="Revenue", value=39000000000.0, date="2025"),
]

for claim in cases:
    v = verify_claim(claim)
    print(claim.text, "->", v.verdict, "|", v.source, "|", (v.quote or "")[:80])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000. -> VERIFIED | Apple Inc. (AAPL) — Revenue | Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 20
Apple's revenue for fiscal year ending 2025-09-27 was $250,000,000,000. -> REFUTED | Apple Inc. (AAPL) — Revenue | Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 20
Netflix's revenue for fiscal year 2025 was $39 billion. -> INSUFFICIENT | None | 


All three verdicts are correct:

- **VERIFIED** — matched the exact chunk, same value.
- **REFUTED** — same chunk retrieved; judge caught the value mismatch.
- **INSUFFICIENT** — Netflix isn't in the knowledge base (`TICKERS` covers
  8 companies); judge recognized none of the top-5 chunks cover it.

## `POST /verify` — both pieces wired into one endpoint

`src/api.py` chains `extract_claims` → `verify_claim` per claim, returning
verdicts. Called here via FastAPI's `TestClient` — no `uvicorn` needed.

In [10]:
from fastapi.testclient import TestClient
from src.api import app

client = TestClient(app)

In [12]:
import json

resp = client.post("/verify", json={"text": "Apple reported revenue of $416,161,000,000 for fiscal year ending 2025-09-27."})
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

200
{
  "claims": [
    {
      "claim": "Apple reported revenue of $416,161,000,000 for fiscal year ending 2025-09-27.",
      "verdict": "VERIFIED",
      "source": "Apple Inc. (AAPL) \u2014 Revenue",
      "quote": "Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 2025-09-27 (10-K filed 2025-10-31)."
    }
  ]
}


**Latency note:** free-tier providers aren't latency-consistent (this call
took under a minute). `src/llm.py` sets a per-call `timeout` with one retry,
so a stuck request fails fast instead of hanging on the client's 600s
default.

## Comparing providers on this notebook's use cases

Testing all three configured providers (OpenRouter, Groq, Ollama) on the
same use cases above — plain call, single-object structured output,
multi-item structured output — with identical code, no per-provider tuning.

In [13]:
from src.config import LLM_PROVIDERS

providers_to_test = [
    (name, cfg["default_model"], cfg["base_url"], cfg["api_key"])
    for name, cfg in LLM_PROVIDERS.items()
]
[(name, model) for name, model, _, _ in providers_to_test]

[('openrouter', 'poolside/laguna-xs-2.1:free'),
 ('groq', 'qwen/qwen3.6-27b'),
 ('ollama', 'ornith:latest')]

### Use case 1 — a plain call (course lesson [`07-llm.md`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/07-llm.md) style)

Free-text answer to a system + user prompt — checks each provider/model
responds sensibly.

In [14]:
from langchain_openai import ChatOpenAI

for name, model, base_url, api_key in providers_to_test:
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, temperature=0)
    try:
        answer = llm.invoke([
            ("system", "Answer in one short sentence."),
            ("user", "What is 15% of 240?"),
        ])
        print(f"{name} ({model}): {answer.content[:200]!r}")
    except Exception as e:
        print(f"{name} ({model}): FAILED — {str(e)[:150]}")

openrouter (poolside/laguna-xs-2.1:free): '15% of 240 is 36.'
groq (qwen/qwen3.6-27b): '\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Question: "What is 15% of 240?"\n   - Constraint: "Answer in one short sentence."\n\n2.  **Calculate the Answer:**\n   - 15% of 240 = '
ollama (ornith:latest): '15% of 240 is 36.'


### Use case 2 — structured output, one object (`Claim`)

Same `Claim` model used in `verifier.py` above: one entity, one value.

In [15]:
for name, model, base_url, api_key in providers_to_test:
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, temperature=0)
    try:
        result = llm.with_structured_output(Claim).invoke(
            "Apple revenue was $394B in FY2023."
        )
        print(f"{name} ({model}): OK — {result}")
    except Exception as e:
        print(f"{name} ({model}): FAILED — {str(e)[:150]}")

openrouter (poolside/laguna-xs-2.1:free): OK — text="Apple's reported revenue for FY2023 was **$383 billion**, not $394 billion. This figure represents the total revenue for the fiscal year ending September 30, 2023. Here's a detailed breakdown of Apple's FY2023 performance:\n\n### Key Revenue Breakdown:\n1. **iPhone**: $205.5 billion (53.7% of total revenue)\n2. **Services**: $80.3 billion (20.9% of total revenue)\n3. **Mac**: $28.5 billion (7.4% of total revenue)\n4. **iPad**: $23.5 billion (6.1% of total revenue)\n5. **Wearables, Home and Accessories**: $35.3 billion (9.2% of total revenue)\n6. **Other Products**: $9.9 billion (2.6% of total revenue)\n\n### Key Highlights:\n- **Growth Drivers**: Strong iPhone 14 sales, continued expansion of the Services ecosystem (including App Store, iCloud, and Apple Music), and growth in Wearables (e.g., Apple Watch and AirPods).\n- **Challenges**: Supply chain constraints, economic headwinds in key markets, and increased competition in the smar

### Use case 3 — structured output, multiple items (`ClaimList`, the shape `extract_claims()` uses)

Same `report_text` from the top, three claims to pull out — checked
separately since "works for one field" doesn't guarantee "works for a list".

In [16]:
from src.claim_extractor import ClaimList, SYSTEM_PROMPT

for name, model, base_url, api_key in providers_to_test:
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, temperature=0)
    try:
        result = llm.with_structured_output(ClaimList).invoke(
            [("system", SYSTEM_PROMPT), ("user", report_text)]
        )
        print(f"{name} ({model}): OK — {len(result.claims)} claims")
    except Exception as e:
        print(f"{name} ({model}): FAILED — {str(e)[:150]}")

openrouter (poolside/laguna-xs-2.1:free): OK — 3 claims
groq (qwen/qwen3.6-27b): FAILED — Error code: 400 - {'error': {'message': 'This model does not support response format `json_schema`. See supported models at https://console.groq.com/d
ollama (ornith:latest): OK — 3 claims


### Takeaway

OpenRouter and Ollama pass all three cases; Groq (`qwen/qwen3.6-27b`) fails
both structured cases (no `json_schema` support in
`with_structured_output()`). Pick a provider/model that passes all three —
swap via `LLM_MODEL` in `.env`, no client code changes needed.

## Orchestration — Airflow DAG for daily ingestion

Phase 1 built the knowledge base once, by hand. In production, sources
change (new filings, daily FRED updates), so ingestion must re-run on a
schedule or the vector store goes stale.

**Input:** `@daily` trigger + `.env` secrets; each `fetch_*` uses dlt
`write_disposition="merge"` (idempotent). **Output:** refreshed `*_raw`
schemas, then a full rebuild of the vector store.

### Design: orchestration only, no reimplemented logic

`dags/fact_checker_dag.py` imports each ingest module's existing `run()`
rather than reimplementing fetch/chunk/embed logic:

```python
from ingest import build_vector_store, fetch_fred, fetch_secedgar, fetch_wikipedia, fetch_worldbank

FETCH_MODULES = {
    "wikipedia": fetch_wikipedia,
    "worldbank": fetch_worldbank,
    "fred": fetch_fred,
    "secedgar": fetch_secedgar,
}

with DAG("fact_checker_daily_ingestion", schedule="@daily", ...) as dag:
    fetch_tasks = [
        PythonOperator(task_id=f"fetch_{name}", python_callable=module.run)
        for name, module in FETCH_MODULES.items()
    ]
    rebuild_vector_store = PythonOperator(
        task_id="rebuild_vector_store", python_callable=build_vector_store.run
    )
    fetch_tasks >> rebuild_vector_store
```

Principles: single responsibility, open/closed (new source = one module +
one dict entry), dependency inversion (DAG depends on each module's
`run()`, not dlt/psycopg2 internals), and a dependency graph matching
reality — fetch tasks are independent; only the rebuild waits on all four,
since it always rebuilds from scratch.

**Runtime is separate from the app.** `Dockerfile.airflow` installs only
dlt, psycopg2, pgvector, sentence-transformers — not langchain/ragas/fastapi.
`docker-compose.yml` runs Airflow standalone (SQLite metadata db, no second
Postgres needed).

In [18]:
import subprocess

# Airflow runs in its own container (`docker compose up -d airflow`), not
# in this notebook's kernel — so we shell out to check it instead of importing
# the DAG module directly (that would try to import `airflow` into this env).
result = subprocess.run(
    ["docker", "exec", "fact-checker-airflow", "airflow", "dags", "list"],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)

dag_id                       | fileloc                               | owners  | is_paused
=============================+=======================================+=========+==========
fact_checker_daily_ingestion | /opt/airflow/dags/fact_checker_dag.py | airflow | False    
                                                                                          



`airflow dags list` shows the DAG with no import errors. Verified
separately: `airflow tasks list` lists all 5 tasks, and a manual
`airflow tasks test ... fetch_wikipedia` run completed with `SUCCESS`.

## What's next

Phase 2 is done: claim extraction, verdict logic, `POST /verify`, Airflow
DAG. Next: hybrid search, reranking, and RAGAS evaluation — see
[docs/phase-3-evaluation.md](../docs/phase-3-evaluation.md) and
[notebooks/phase3_evaluation.ipynb](phase3_evaluation.ipynb).

[← Back to README](../README.md)